In [1]:
import logging
import re
import time
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional
from urllib.parse import urljoin, urlparse

import pandas as pd

import requests
from bs4 import BeautifulSoup

!pip install pro-football-reference-web-scraper
import pro_football_reference_web_scraper


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
REQUEST_DELAY_SECONDS = 1.5

# Minimum usable content length — shorter than this is likely a paywall / error page
MIN_CONTENT_CHARS = 200

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8",
    "Referer": "https://www.google.com/",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Accept-Encoding": "gzip, deflate, br",
}

# HEADERS = {
#         'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
#     }

player_name = "Josh Allen"
pfr_player_id = "AlleJo02"
season = 2023

first_letter = pfr_player_id[0].upper()
url = f"https://www.pro-football-reference.com/players/{first_letter}/{pfr_player_id}/gamelog/{season}"
print(url)

https://www.pro-football-reference.com/players/A/AlleJo02/gamelog/2023


In [3]:
# from pro_football_reference_web_scraper import player_game_log as p
# from pro_football_reference_web_scraper import team_game_log as t

# player_log = p.get_player_game_log(player="Josh Allen", position="QB", season=2020)

In [4]:
resp = requests.get(url, headers=HEADERS, timeout=15)
# resp.raise_for_status()
soup = BeautifulSoup(resp.text, "lxml")

In [89]:
valid_positions = ['QB', 'RB', 'WR', 'TE']


# function that returns a player's game log in a given season
# player: player's full name (e.g. Tom Brady)
# position: abbreviation (QB, RB, WR, TE only)
def get_player_game_log(player: str, pfr_id: str, position: str, season: int) -> pd.DataFrame:
    """A function to retrieve a player's game log in a given season.

    Returns a pandas DataFrame of a NFL player's game log in a given season, including position-specific statistics.

    Args:
        player (str): A NFL player's full name, as it appears on Pro Football Reference
        position (str): The position the player plays. Must be 'QB', 'RB', 'WR', or 'TE'
        season (int): The season of the game log you are trying to retrieve

    Returns:
        pandas.DataFrame: Each game is a row of the DataFrame

    """

    # position arg must be formatted properly
    if position not in valid_positions:
        raise Exception('Invalid position: "position" arg must be "QB", "RB", "WR", or "TE"')

    # # make request to find proper href
    r1 = make_request_list(player, pfr_id, position, season)
    player_list = get_soup(r1)
    print(player_list.text)
    
    # # find href
    href = get_href(player, position, season, player_list)

    # make HTTP request and extract HTML
    r2 = make_request_player(href, season)

    # parse HTML using BeautifulSoup
    game_log = get_soup(r2)

    # generating the appropriate game log format according to position
    if 'QB' in position:
        return qb_game_log(game_log)
    elif 'WR' in position or 'TE' in position:
        return wr_game_log(game_log, season)
    elif 'RB' in position:
        return rb_game_log(game_log)

# helper function that makes a HTTP request over a list of players with a given last initial
def make_request_list(player: str, pfr_player_id: str, position: str, season: int):
    # name_split = player.split(' ')
    # last_initial = name_split[1][0]
    # url = 'https://www.pro-football-reference.com/players/%s/' % (last_initial)
    first_letter = pfr_player_id[0].upper()
    url = f"https://www.pro-football-reference.com/players/{first_letter}/{pfr_player_id}/gamelog/{season}"
    print(f"Requesting from {url}")
    return requests.get(url, headers=HEADERS, timeout=15)


# helper function that makes a HTTP request for a given player's game log
def make_request_player(href: str, season: int):
    # url = 'https://www.pro-football-reference.com%s/gamelog/%s/' % (href, season)
    # return requests.get(url)
    first_letter = pfr_player_id[0].upper()
    url = f"https://www.pro-football-reference.com/players/{first_letter}/{pfr_player_id}/gamelog/{season}"
    print(f"Requesting from {url}")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
    }
    return requests.get(url, headers=HEADERS, timeout=15)
    # return requests.get(url, headers=headers, timeout=15)


# helper function that takes a requests.Response object and returns a BeautifulSoup object
def get_soup(request):
    # return BeautifulSoup(request.text, 'html.parser')
    return BeautifulSoup(request.text, 'lxml')

# helper function that takes a BeautifulSoup object and converts it into a pandas dataframe containing a QB game log
def qb_game_log(soup: BeautifulSoup) -> pd.DataFrame:
    # Most relevant QB stats, in my opinion. Could adjust if necessary
    data = {
        'date': [],
        'week': [],
        'team': [],
        'game_location': [],
        'opp': [],
        'result': [],
        'team_pts': [],
        'opp_pts': [],
        'cmp': [],
        'att': [],
        'pass_yds': [],
        'pass_td': [],
        'int': [],
        'rating': [],
        'sacked': [],
        'rush_att': [],
        'rush_yds': [],
        'rush_td': [],
    }  # type: dict

    table_rows = soup.find('tbody').find_all('tr')

    # ignore inactive or DNP games
    to_ignore = []
    for i in range(len(table_rows)):
        elements = table_rows[i].find_all('td')
        x = elements[len(elements) - 1].text
        if x == 'Inactive' or x == 'Did Not Play' or x == 'Injured Reserve':
            to_ignore.append(i)

    # adding data to data dictionary
    for i in range(len(table_rows)):
        if i not in to_ignore:
            data['date'].append(table_rows[i].find('td', {'data-stat': 'game_date'}).text)
            data['week'].append(int(table_rows[i].find('td', {'data-stat': 'week_num'}).text))
            data['team'].append(table_rows[i].find('td', {'data-stat': 'team'}).text)
            data['game_location'].append(table_rows[i].find('td', {'data-stat': 'game_location'}).text)
            data['opp'].append(table_rows[i].find('td', {'data-stat': 'opp'}).text)
            data['result'].append(table_rows[i].find('td', {'data-stat': 'game_result'}).text.split(' ')[0])
            data['team_pts'].append(
                int(table_rows[i].find('td', {'data-stat': 'game_result'}).text.split(' ')[1].split('-')[0])
            )
            data['opp_pts'].append(
                int(table_rows[i].find('td', {'data-stat': 'game_result'}).text.split(' ')[1].split('-')[1])
            )
            data['cmp'].append(int(table_rows[i].find('td', {'data-stat': 'pass_cmp'}).text)) if table_rows[i].find(
                'td', {'data-stat': 'pass_cmp'}
            ).text != '' else data['cmp'].append(0)
            data['att'].append(int(table_rows[i].find('td', {'data-stat': 'pass_att'}).text)) if table_rows[i].find(
                'td', {'data-stat': 'pass_att'}
            ).text != '' else data['att'].append(0)
            data['pass_yds'].append(int(table_rows[i].find('td', {'data-stat': 'pass_yds'}).text)) if table_rows[
                i
            ].find('td', {'data-stat': 'pass_yds'}).text != '' else data['pass_yds'].append(0)
            data['pass_td'].append(int(table_rows[i].find('td', {'data-stat': 'pass_td'}).text)) if table_rows[i].find(
                'td', {'data-stat': 'pass_td'}
            ).text != '' else data['pass_td'].append(0)
            data['int'].append(int(table_rows[i].find('td', {'data-stat': 'pass_int'}).text)) if table_rows[i].find(
                'td', {'data-stat': 'pass_int'}
            ).text != '' else data['int'].append(0)
            data['rating'].append(float(table_rows[i].find('td', {'data-stat': 'pass_rating'}).text)) if table_rows[
                i
            ].find('td', {'data-stat': 'pass_rating'}).text != '' else data['rating'].append(0)
            data['sacked'].append(int(table_rows[i].find('td', {'data-stat': 'pass_sacked'}).text)) if table_rows[
                i
            ].find('td', {'data-stat': 'pass_sacked'}).text != '' else data['sacked'].append(0)
            data['rush_att'].append(int(table_rows[i].find('td', {'data-stat': 'rush_att'}).text)) if table_rows[
                i
            ].find('td', {'data-stat': 'rush_att'}).text != '' else data['rush_att'].append(0)
            data['rush_yds'].append(int(table_rows[i].find('td', {'data-stat': 'rush_yds'}).text)) if table_rows[
                i
            ].find('td', {'data-stat': 'rush_yds'}).text != '' else data['rush_yds'].append(0)
            data['rush_td'].append(int(table_rows[i].find('td', {'data-stat': 'rush_td'}).text)) if table_rows[i].find(
                'td', {'data-stat': 'rush_td'}
            ).text != '' else data['rush_td'].append(0)

    return pd.DataFrame(data=data)


# helper function that takes a BeautifulSoup object and converts it into a pandas dataframe containing a WR/TE game log
def wr_game_log(soup: BeautifulSoup, season: int) -> pd.DataFrame:
    # Most relevant WR stats, in my opinion.
    # Could adjust if necessary (maybe figure out how to incorporate rushing stats?)

    data = {
        'date': [],
        'week': [],
        'team': [],
        'game_location': [],
        'opp': [],
        'result': [],
        'team_pts': [],
        'opp_pts': [],
        'tgt': [],
        'rec': [],
        'rec_yds': [],
        'rec_td': [],
        'snap_pct': [],
    }  # type: dict

    table_rows = soup.find('tbody').find_all('tr')

    # ignore inactive or DNP games
    to_ignore = []
    for i in range(len(table_rows)):
        elements = table_rows[i].find_all('td')
        x = elements[len(elements) - 1].text
        if x == 'Inactive' or x == 'Did Not Play' or x == 'Injured Reserve':
            to_ignore.append(i)

    # adding data to data dictionray
    for i in range(len(table_rows)):
        if i not in to_ignore:
            data['date'].append(table_rows[i].find('td', {'data-stat': 'game_date'}).text)
            data['week'].append(int(table_rows[i].find('td', {'data-stat': 'week_num'}).text))
            data['team'].append(table_rows[i].find('td', {'data-stat': 'team'}).text)
            data['game_location'].append(table_rows[i].find('td', {'data-stat': 'game_location'}).text)
            data['opp'].append(table_rows[i].find('td', {'data-stat': 'opp'}).text)
            data['result'].append(table_rows[i].find('td', {'data-stat': 'game_result'}).text.split(' ')[0])
            data['team_pts'].append(
                int(table_rows[i].find('td', {'data-stat': 'game_result'}).text.split(' ')[1].split('-')[0])
            )
            data['opp_pts'].append(
                int(table_rows[i].find('td', {'data-stat': 'game_result'}).text.split(' ')[1].split('-')[1])
            )
            data['tgt'].append(int(table_rows[i].find('td', {'data-stat': 'targets'}).text))
            data['rec'].append(int(table_rows[i].find('td', {'data-stat': 'rec'}).text))
            data['rec_yds'].append(int(table_rows[i].find('td', {'data-stat': 'rec_yds'}).text))
            data['rec_td'].append(int(table_rows[i].find('td', {'data-stat': 'rec_td'}).text))
            if season > 2011:
                data['snap_pct'].append(float(int(table_rows[i].find('td', {'data-stat': 'off_pct'}).text[:-1]) / 100))
            else:
                data['snap_pct'].append('Not Available')

    return pd.DataFrame(data=data)


def rb_game_log(soup: BeautifulSoup) -> pd.DataFrame:
    # Most relevant RB stats, in my opinion. Could adjust if necessary
    data = {
        'date': [],
        'week': [],
        'team': [],
        'game_location': [],
        'opp': [],
        'result': [],
        'team_pts': [],
        'opp_pts': [],
        'rush_att': [],
        'rush_yds': [],
        'rush_td': [],
        'tgt': [],
        'rec': [],
        'rec_yds': [],
        'rec_td': [],
    }  # type: dict

    table_rows = soup.find('tbody').find_all('tr')

    # ignore inactive or DNP games
    to_ignore = []
    for i in range(len(table_rows)):
        elements = table_rows[i].find_all('td')
        x = elements[len(elements) - 1].text
        if x == 'Inactive' or x == 'Did Not Play' or x == 'Injured Reserve':
            to_ignore.append(i)

    # adding data to data dictionary
    for i in range(len(table_rows)):
        if i not in to_ignore:
            data['date'].append(table_rows[i].find('td', {'data-stat': 'game_date'}).text)
            data['week'].append(int(table_rows[i].find('td', {'data-stat': 'week_num'}).text))
            data['team'].append(table_rows[i].find('td', {'data-stat': 'team'}).text)
            data['game_location'].append(table_rows[i].find('td', {'data-stat': 'game_location'}).text)
            data['opp'].append(table_rows[i].find('td', {'data-stat': 'opp'}).text)
            data['result'].append(table_rows[i].find('td', {'data-stat': 'game_result'}).text.split(' ')[0])
            data['team_pts'].append(
                int(table_rows[i].find('td', {'data-stat': 'game_result'}).text.split(' ')[1].split('-')[0])
            )
            data['opp_pts'].append(
                int(table_rows[i].find('td', {'data-stat': 'game_result'}).text.split(' ')[1].split('-')[1])
            )
            data['rush_att'].append(int(table_rows[i].find('td', {'data-stat': 'rush_att'}).text)) if table_rows[
                i
            ].find('td', {'data-stat': 'rush_att'}).text != '' else data['rush_att'].append(0)
            data['rush_yds'].append(int(table_rows[i].find('td', {'data-stat': 'rush_yds'}).text)) if table_rows[
                i
            ].find('td', {'data-stat': 'rush_yds'}).text != '' else data['rush_yds'].append(0)
            data['rush_td'].append(int(table_rows[i].find('td', {'data-stat': 'rush_td'}).text)) if table_rows[i].find(
                'td', {'data-stat': 'rush_td'}
            ).text != '' else data['rush_td'].append(0)
            data['tgt'].append(int(table_rows[i].find('td', {'data-stat': 'targets'}).text)) if table_rows[i].find(
                'td', {'data-stat': 'targets'}
            ).text != '' else data['tgt'].append(0)
            data['rec'].append(int(table_rows[i].find('td', {'data-stat': 'rec'}).text)) if table_rows[i].find(
                'td', {'data-stat': 'rec'}
            ).text != '' else data['rec'].append(0)
            data['rec_yds'].append(int(table_rows[i].find('td', {'data-stat': 'rec_yds'}).text)) if table_rows[i].find(
                'td', {'data-stat': 'rec_yds'}
            ).text != '' else data['rec_yds'].append(0)
            data['rec_td'].append(int(table_rows[i].find('td', {'data-stat': 'rec_td'}).text)) if table_rows[i].find(
                'td', {'data-stat': 'rec_td'}
            ).text != '' else data['rec_td'].append(0)

    return pd.DataFrame(data=data)

# helper function that gets the player's href
def get_href(player: str, position: str, season: int, player_list: BeautifulSoup) -> str:
    print(player_list)
    # players = player_list.find('div', id='all_stats', recursive=True)
    # table = player_list.find('table', id='stats', recursive=False)
    # print(table)
    # print(players)
    # players = players.find_all('p')
    # for p in players:
    #     seasons = p.text.split(' ')
    #     seasons = seasons[len(seasons) - 1].split('-')
    #     if season >= int(seasons[0]) and season <= int(seasons[1]) and player in p.text and position in p.text:
    #         return p.find('a').get('href').replace('.htm', '')
    # raise Exception('Cannot find a ' + position + ' named ' + player + ' from ' + str(season))

In [90]:
get_player_game_log(player_name, pfr_player_id, position="QB", season=2023)

Requesting from https://www.pro-football-reference.com/players/A/AlleJo02/gamelog/2023
Just a moment...Enable JavaScript and cookies to continue
<!DOCTYPE html>
<html lang="en-US"><head><title>Just a moment...</title><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/><meta content="IE=Edge" http-equiv="X-UA-Compatible"/><meta content="noindex,nofollow" name="robots"/><meta content="width=device-width,initial-scale=1" name="viewport"/><meta content="default-src 'none'; script-src 'nonce-4w4RXjxGBDjBsdzhg3Jyvu' 'unsafe-eval' https://challenges.cloudflare.com; script-src-attr 'none'; style-src 'unsafe-inline'; img-src 'self' https://challenges.cloudflare.com; connect-src 'self' https://challenges.cloudflare.com; frame-src 'self' https://challenges.cloudflare.com blob:; child-src 'self' https://challenges.cloudflare.com blob:; worker-src blob:; form-action http: https:; base-uri 'self'" http-equiv="content-security-policy"/><style>*{box-sizing:border-box;margin:0;padding:0

AttributeError: 'NoneType' object has no attribute 'find_all'

In [ ]:
.find('body')

In [ ]:
<body>
    <div class="main-wrapper" role="main">
        <div class="main-content"><noscript>
            <div class="h2">
                <span id="challenge-error-text">Enable JavaScript and cookies to continue</span>
            </div>
            </noscript>
        </div>
    </div>
    <script nonce="cTPadbGEY3S7HVbz1v7Ewm">(function(){window._cf_chl_opt = {cFPWv: 'g',cH: 'UQ4SqDaCq5qUJ5uVr4Al.IoUe6rPOmRBhygJnfqOHzM-1780880104-1.2.1.1-Go9fT9Fu7TIsK4vOs0_zsPDXsMa6jKR0lH80DhcZS9jEWcapewvwyPnHFTSzZLEX',cITimeS: '1780880104',cN: 'cTPadbGEY3S7HVbz1v7Ewm',cRay: 'a083edcb5fb8f78d',cTplB: '0',cTplC:0,cTplO:0,cTplV:5,cType: 'managed',cUPMDTk:"/players/A/AlleJo02/gamelog/2023?__cf_chl_tk=rzJuCiEe7G.vvnHcDA9J08dQMi5H74BHungdE.3Mojk-1780880104-1.0.1.1-IdIWGE.sZ7Ic635Dr6qhGmM4mg4qJ.qDpnzESzmLhVA",cvId: '3',cZone: 'www.pro-football-reference.com',fa:"/players/A/AlleJo02/gamelog/2023?__cf_chl_f_tk=rzJuCiEe7G.vvnHcDA9J08dQMi5H74BHungdE.3Mojk-1780880104-1.0.1.1-IdIWGE.sZ7Ic635Dr6qhGmM4mg4qJ.qDpnzESzmLhVA",md: 'xLtBms4Kq5VyPQUEfv5HxvctP.HoIUOgTwpkIVa3LbI-1780880104-1.2.1.1-jM0IUnjGU1_1zlZ3DHisiU1E6DrWBeFD_QiJLjUFtqE1uZJJzEqsagT6GFOvfdQsdVA_cYyHkpgOPhxDkAvTEiHV7iBTVWl5u9ghoiHZwx9409Lg6mGxrxrK3pfuNsfrVY5sUtdULHy52BwProUuNGOV.ZYBgeuoZq0k_25kaJ_PLOg1lDOBvmMjcdg0wfQFR2RwHIsY4F.xftyHOiz9zkLVVzUbBU3MFvYnAqsFkXrc1.zoeIypwlWhSK2oEaFf2p.UQOhkY9EWV4e_BXq5Z89Zq9Y0tJPrzkNxKE8fei2YbmDsWPEQ0aG_Jc.ln7Q4MY8ZLR59fKMeCx5jTED2BYGLiiC8K7Dh5OCRADDKHphoXqe5pCRO8W7Qko5N5aP5ZTuyxWGyFnBw6pjV4t_IVeghKfI4htJYbFIb0R_6Zr2Y1dkCT01mpBClgDZcPFgJOYg_4j12oySV51YjVljEOP9rRPsyJZxguk_ynvxS5uS_yLhwQg08wevWLmu8cdkASnCv6BxRZt9y4AmUkH4qQusa3b41vFLEN13tO.SyUviQ.2gZChGHwFMeEmZPu2U63._i153OVHyXPYhlextOhOSvZoVA1cX66pDTKiR4yqWn1nkzjS9QiutKivxvfr2w4MfNNjfwp36INS6b9nTe4EBxdN_EXT8TwyRlv2ZyZE5NafXGRkbFxnTZzOl_5JzU65.Im2eFDplwrwUUa.YyAtNRXcgFVkLapdfjprUfSWTbn3bPDGn3XcHUq9cVPkX2.qAp4p6nyDlf8NAuCQDdxAaFqMtwHBr65HfelpW8nMS3YqjjHoNIG5DNkk_UtDx.QnJhuZ0sYj4soCYdz5a67rt84elA5uAHibgnbdOgCZ1I.cYcLKN5RhydNAsD3XvHHPXOEYdhkFlE4Z3pI0ZsGAxR_Ry3lPiBcgkku1nFGiyXcNIbrAl0O1iYyfxfBgsYwns.K_IOsXZRGj_6y_PlKIzvCmTle73EIhx.eWr84o1NgDAWberyfsc7SnoNthZgp5FnuDtzEmf9nCbbJ4IPGLJJpwtJcTXxMvxYW6_CFDArK6ipbaJOb8W5xG23B2DapGec6uo10TTRfAddLIGVhviAPgcKum.3fqhRlHD.1Vu12hYGS7DIK7fgBYuc3Q3U9QY14Llg7gXx8S_hxSFM6XxkA6iB.QUctHNRtdoJDf2.MBH2qNHHNIXInTlO.u.lX8u2o19YVuvQlYeMjjJkigGPLE3ZUpOPiEe8VGyMUjUHgxNh5iQedsZo_7ZzyKHtNTANfA6JDYyvYHpo_3N9v5bnVFi.XTxWNNsvmqs9jHCRSPEqngWbXnEtya5AV7tE',mdrd: 'GeGvWVK4jrvmKxj__Bfdh2SJw4sKYBkNW8o29LSOu5c-1780880104-1.2.1.1-q4ardgIZOyPpYeBfslYZzyesVg3riGAnAv8zAdQE2j5SzFV3rBZGFMmk1PSxaa09e77R4eLubl8bwrXO.fqR6yw86w7OVTNM2Md08DLYP11vZtRczBIXpd5iGbEg2Pmnw.15TqLu3ss4nkrSUjinJDXlqUpMsdFntHVVdDqmM.jFZtpqjCwBJUvOkEwKXhIHuEF2YxlGD54J.R6OnnsCh0bNyqYEuInci0OMx77X1y_QBeizZyrPAc_u_42CI0VgHh0xOX79In3UG8UM4TjkJCsIql7XkSJKL7T2vjPnfSgeLIprIUDFoJVXe11MWvW28AapbyovL44lgN_Tw35g.bL4PvCsuKzIi_HJD0G2HuxVPcAkmlsrUu5IcuAmMbMdgmJmDWncQ1f1bmliJAEk6vI_o5mSCDl4.EMeqBUt6BDx1YoBPU3_FoBVp1m0a8HdTLpEAGs71JfILPXjnZNzuEvUmenHXRmmvwlQ_i.Pyq2fAGMxOwwh_m1V8tMaj9aEDwKTOgsINE2a6QcD5OYCvXl0_XbSbvDLkRns0WBd3WY',};var a = document.createElement('script');a.nonce = 'cTPadbGEY3S7HVbz1v7Ewm';a.src = '/cdn-cgi/challenge-platform/h/g/orchestrate/chl_page/v1?ray=a083edcb5fb8f78d';window._cf_chl_opt.cOgUHash = location.hash === '' && location.href.indexOf('#') !== -1 ? '#' : location.hash;window._cf_chl_opt.cOgUQuery = location.search === '' && location.href.slice(0, location.href.length - window._cf_chl_opt.cOgUHash.length).indexOf('?') !== -1 ? '?' : location.search;if (window.history && window.history.replaceState) {var ogU = location.pathname + window._cf_chl_opt.cOgUQuery + window._cf_chl_opt.cOgUHash;history.replaceState(null, null,"/players/A/AlleJo02/gamelog/2023?__cf_chl_rt_tk=rzJuCiEe7G.vvnHcDA9J08dQMi5H74BHungdE.3Mojk-1780880104-1.0.1.1-IdIWGE.sZ7Ic635Dr6qhGmM4mg4qJ.qDpnzESzmLhVA"+ window._cf_chl_opt.cOgUHash);a.onload = function() {history.replaceState(null, null, ogU);}}document.getElementsByTagName('head')[0].appendChild(a);}());
    </script>
</body>
